# DupHunter Deduplication Simulation

This notebook simulates the deduplication logic described in the DupHunter paper (USENIX ATC '20).

## Data Flow
1. **Input**: A directory containing compressed layer tarballs (`.tar.gz` or gzip-compressed files).
2. **Decompress & Extract**: Each layer is fully decompressed (gzip) and unpacked (tar) to extract individual files into a specified output directory.
3. **File-Level Dedup (GF-R)**: Compute SHA-256 hash per extracted file, remove exact duplicates globally.
4. **Block-Level Dedup (512KiB)**: Unique files are further split into **512KiB fixed-size blocks**, block-level dedup via SHA-256.

> **Note**: DupHunter does NOT use partial decompression (that is SimEnc's technique).  
> Layers must be **fully decompressed and unpacked** before deduplication.

# DupHunter Deduplication Ratio Analysis

## Deduplication Modes Tested

| Mode | Description |
|------|-------------|
| **GF-R (File-Level)** | SHA-256 hash per file, remove exact duplicate files globally |
| **GF+Block-R (File + 512KiB Block)** | File-level dedup first, then split unique files into 512KiB fixed blocks and remove duplicate blocks |

## Cross-Method Comparison

| Method | Granularity | Block Size | Clustering |
|--------|------------|------------|------------|
| **DupHunter GF-R** | File-level (SHA-256) | N/A | None |
| **DupHunter GF+Block-R** | File + Block-level | 512KiB fixed | None |
| **MLE** | Chunk-level (SHA-256) | 512KiB | None |
| **Finesse (LSH)** | Chunk-level (LSH) | fastcdc variable | LSH clusters |
| **SimEnc** | Chunk-level (Semantic hash) | fastcdc variable | DBSCAN/KMeans |

## 0. Import Libraries

In [ ]:
import os
import gzip
import tarfile
import hashlib
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from multiprocessing import Pool
from collections import defaultdict
import time

## 1. Configure Paths

- `layer_directory`: Directory containing the compressed layer blobs (gzip-compressed tar files).
- `extract_directory`: Directory where decompressed and extracted files will be stored.
- Only files larger than 100KB are considered as valid layers (to skip config/manifest files).

In [ ]:
def find_files(directory):
    """Recursively find all files in a directory."""
    for root, _, files in os.walk(directory):
        for file in files:
            yield os.path.join(root, file)


# ===== 修改为你的 layer 数据目录 =====
layer_directory = '/home/user/data/layer_data_ubuntu/'
# ===== 解压后文件的输出目录 =====
extract_directory = '/home/user/data/duphunter_extracted_files/'

# Pool size for multiprocessing
pool_size = 30

# Scan layer directory for compressed layer blobs
all_layers = list(find_files(layer_directory))
# Filter: only keep files > 100KB (skip config/manifest small files)
all_layers = [f for f in all_layers if os.path.getsize(f) > 100 * 1024]

print(f'Layer directory: {layer_directory}')
print(f'Extract directory: {extract_directory}')
print(f'Total layer files found: {len(all_layers)}')
if all_layers:
    print(f'Example layer: {all_layers[0]}')
    total_compressed = sum(os.path.getsize(f) for f in all_layers)
    print(f'Total compressed size: {total_compressed/1024/1024/1024:.4f} GiB')

---
## 2. Decompress & Extract Layers

DupHunter operates on **individual files** extracted from container image layers.
Each layer is a gzip-compressed tar archive. We:
1. **gzip decompress** the layer blob.
2. **tar extract** the resulting archive to get individual files.
3. Each layer is extracted into its own subdirectory (named by layer digest) to preserve file paths.

This corresponds to DupHunter's push-time preprocessing:
```
compressed layer (.tar.gz) → gzip -d → .tar → tar xf → individual files
```

In [ ]:
def extract_layer(layer_path, output_base_dir):
    """
    Decompress (gzip) and extract (tar) a single layer to output directory.
    Each layer gets its own subdirectory named by a hash of the layer path.
    Returns (layer_path, layer_subdir, num_files, status).
    """
    # Create a unique subdirectory for this layer
    layer_id = hashlib.md5(layer_path.encode()).hexdigest()
    layer_subdir = os.path.join(output_base_dir, layer_id)
    
    try:
        os.makedirs(layer_subdir, exist_ok=True)
        
        # Try to open as gzip-compressed tar
        with tarfile.open(layer_path, 'r:gz') as tar:
            # Security: filter out absolute paths and '..' to prevent path traversal
            members = []
            for member in tar.getmembers():
                # Skip directories, only extract regular files
                if member.isfile() and member.size > 0:
                    # Sanitize path
                    member.name = os.path.normpath(member.name).lstrip(os.sep)
                    if not member.name.startswith('..'):
                        members.append(member)
            tar.extractall(path=layer_subdir, members=members)
        
        num_files = len(members)
        return layer_path, layer_subdir, num_files, 'OK'
    
    except tarfile.ReadError:
        # Not a valid tar.gz, try plain gzip → then treat the decompressed content as a single file
        try:
            out_file = os.path.join(layer_subdir, 'decompressed_blob')
            with gzip.open(layer_path, 'rb') as f_in:
                with open(out_file, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
            return layer_path, layer_subdir, 1, 'GZIP_ONLY'
        except Exception as e:
            return layer_path, layer_subdir, 0, f'ERROR: {e}'
    
    except Exception as e:
        return layer_path, layer_subdir, 0, f'ERROR: {e}'


# ===== Decompress all layers =====
os.makedirs(extract_directory, exist_ok=True)

print(f'Decompressing {len(all_layers)} layers to: {extract_directory}')
print('This may take a while...\n')

start_time = time.time()
extract_results = []

for i, layer_path in enumerate(all_layers, 1):
    result = extract_layer(layer_path, extract_directory)
    extract_results.append(result)
    if i % 100 == 0 or i == len(all_layers):
        print(f'  Extracted {i}/{len(all_layers)} layers', end='\r')

elapsed = time.time() - start_time

# Statistics
ok_count = sum(1 for r in extract_results if r[3] == 'OK')
gzip_count = sum(1 for r in extract_results if r[3] == 'GZIP_ONLY')
err_count = sum(1 for r in extract_results if r[3].startswith('ERROR'))
total_extracted_files = sum(r[2] for r in extract_results)

print(f'\n\nDecompression complete in {elapsed:.2f}s')
print(f'  tar.gz layers:    {ok_count}')
print(f'  gzip-only layers: {gzip_count}')
print(f'  errors:           {err_count}')
print(f'  Total extracted files: {total_extracted_files}')

In [ ]:
# Scan all extracted files from the decompressed layers
all_extracted_files = list(find_files(extract_directory))
# Filter out empty files (DupHunter skips empty files in CheckDuplicate)
all_extracted_files = [f for f in all_extracted_files if os.path.getsize(f) > 0]

total_extracted_size = sum(os.path.getsize(f) for f in all_extracted_files)

print(f'Total extracted files (non-empty): {len(all_extracted_files)}')
print(f'Total extracted size: {total_extracted_size/1024/1024/1024:.4f} GiB')
if all_extracted_files:
    print(f'Example file: {all_extracted_files[0]}')

In [ ]:
def hash_file_sha256(filename):
    """Compute SHA-256 hash for a file (same as DupHunter's digest computation)."""
    sha256_hash = hashlib.sha256()
    with open(filename, 'rb') as f:
        for byte_block in iter(lambda: f.read(65536), b''):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()


def hash_file_worker(filename):
    """Worker function for multiprocessing."""
    file_size = os.path.getsize(filename)
    file_hash = hash_file_sha256(filename)
    return filename, file_hash, file_size


# Parallel SHA-256 hashing on all extracted files
print(f'Computing SHA-256 hashes for {len(all_extracted_files)} extracted files...')
start_time = time.time()

results = []
with Pool(pool_size) as p:
    for i, result in enumerate(p.imap(hash_file_worker, all_extracted_files), 1):
        results.append(result)
        if i % 10000 == 0:
            print(f'  Processed {i}/{len(all_extracted_files)} files')

elapsed = time.time() - start_time
print(f'\nFinished hashing in {elapsed:.2f}s')

# Build DataFrame
df_files = pd.DataFrame(results, columns=['file_name', 'sha256', 'file_size'])
print(f'Total files: {len(df_files)}')
print(f'Unique hashes: {df_files["sha256"].nunique()}')
df_files.head()

In [ ]:
# --- DupHunter GF-R: File-Level Dedup Statistics ---

total_size = df_files['file_size'].sum()

# Unique files: first occurrence of each hash
unique_files = df_files.drop_duplicates(subset='sha256', keep='first')
unique_size = unique_files['file_size'].sum()

# Deduplicated (removed) size
dedup_size = total_size - unique_size

# Distribution info
hash_counts = df_files['sha256'].value_counts()
dup_hashes = hash_counts[hash_counts > 1]
unique_only_hashes = hash_counts[hash_counts == 1]

dedup_ratio_gfr = total_size / unique_size if unique_size > 0 else float('inf')

print('=' * 60)
print('DupHunter GF-R (File-Level Dedup) Results')
print('=' * 60)
print(f'Total files:                   {len(df_files)}')
print(f'Unique files:                  {len(unique_files)} ({len(unique_files)/len(df_files)*100:.1f}%)')
print(f'Duplicate files removed:       {len(df_files) - len(unique_files)}')
print(f'')
print(f'Total size (decompressed):     {total_size/1024/1024/1024:.4f} GiB')
print(f'After file dedup:              {unique_size/1024/1024/1024:.4f} GiB')
print(f'Saved by file dedup:           {dedup_size/1024/1024/1024:.4f} GiB')
print(f'')
print(f'Dedup ratio (GF-R):            {dedup_ratio_gfr:.4f}x')
print(f'Space saving:                  {dedup_size/total_size*100:.2f}%')
print('=' * 60)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(hash_counts.values, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Copies')
axes[0].set_ylabel('Number of Unique Hashes')
axes[0].set_title('DupHunter: File Duplication Distribution')
axes[0].set_yscale('log')

sns.ecdfplot(data=hash_counts.values, ax=axes[1])
axes[1].set_xlabel('Number of Copies')
axes[1].set_ylabel('CDF')
axes[1].set_title('DupHunter: CDF of File Duplication')

plt.tight_layout()
plt.savefig('./duphunter_file_dedup_distribution.png', dpi=150)
plt.show()

---
## 4. Block-Level Dedup (512KiB Fixed Blocks)

After file-level dedup, we split each **unique file** into **512KiB fixed-size blocks** and perform
block-level deduplication via SHA-256 hashing.

This uses the same 512KiB block granularity as MLE/SimEnc for fair cross-method comparison.

In [ ]:
import multiprocessing

BLOCK_SIZE = 512 * 1024  # 512 KiB

def process_file_blocks(file_name):
    """
    Split a file into 512KiB fixed-size blocks and compute SHA-256 per block.
    The last block may be smaller than 512KiB.
    """
    results = []
    try:
        with open(file_name, 'rb') as f:
            while True:
                block = f.read(BLOCK_SIZE)
                if not block:
                    break
                block_hash = hashlib.sha256(block).hexdigest()
                block_size = len(block)
                results.append([file_name, block_hash, block_size])
    except Exception as e:
        print(f'Error processing {file_name}: {e}')
    return results

In [ ]:
# Get unique files (after file-level dedup)
unique_file_list = unique_files['file_name'].tolist()
print(f'Unique files for block-level dedup: {len(unique_file_list)}')
print(f'Unique files total size: {unique_size/1024/1024/1024:.4f} GiB')
print(f'Block size: {BLOCK_SIZE} bytes ({BLOCK_SIZE//1024} KiB)')

# Split unique files into 512KiB blocks
print('\nSplitting unique files into 512KiB fixed blocks...')
start_time = time.time()

block_results = []
with multiprocessing.Pool(processes=pool_size) as pool:
    for i, result in enumerate(pool.imap(process_file_blocks, unique_file_list), 1):
        block_results.extend(result)
        if i % 10000 == 0:
            print(f'  Processed {i}/{len(unique_file_list)} unique files')

elapsed = time.time() - start_time
print(f'\nFinished block splitting in {elapsed:.2f}s')

df_blocks = pd.DataFrame(block_results, columns=['file_name', 'hash', 'size'])
df_blocks['size'] = df_blocks['size'].astype(int)
print(f'Total 512KiB blocks: {len(df_blocks)}')
print(f'Unique block hashes: {df_blocks["hash"].nunique()}')

In [ ]:
# --- DupHunter GF+Block-R: File + 512KiB Block-Level Dedup Statistics ---

block_hash_stats = df_blocks.groupby('hash')['size'].agg(['sum', 'first', 'count'])
block_hash_stats.columns = ['total_size', 'first_size', 'count']

# After block dedup: keep one copy of each unique block
block_unique_size = block_hash_stats['first_size'].sum()
block_total_size = block_hash_stats['total_size'].sum()
block_dedup_size = block_total_size - block_unique_size

# Combined: file-level saved + block-level saved
total_saved_combined = dedup_size + block_dedup_size
final_stored_size = total_size - total_saved_combined
dedup_ratio_combined = total_size / final_stored_size if final_stored_size > 0 else float('inf')

print('=' * 60)
print('DupHunter GF+Block-R (File + 512KiB Block Dedup) Results')
print('=' * 60)
print(f'Step 1 - File-Level Dedup:')
print(f'  Total size (decompressed):   {total_size/1024/1024/1024:.4f} GiB')
print(f'  After file dedup:            {unique_size/1024/1024/1024:.4f} GiB')
print(f'  Saved by file dedup:         {dedup_size/1024/1024/1024:.4f} GiB')
print(f'')
print(f'Step 2 - 512KiB Block-Level Dedup (on unique files):')
print(f'  Total block size:            {block_total_size/1024/1024/1024:.4f} GiB')
print(f'  After block dedup:           {block_unique_size/1024/1024/1024:.4f} GiB')
print(f'  Saved by block dedup:        {block_dedup_size/1024/1024/1024:.4f} GiB')
print(f'')
print(f'Combined Results:')
print(f'  Original total size:         {total_size/1024/1024/1024:.4f} GiB')
print(f'  Final stored size:           {final_stored_size/1024/1024/1024:.4f} GiB')
print(f'  Total saved:                 {total_saved_combined/1024/1024/1024:.4f} GiB')
print(f'')
print(f'  Dedup ratio (GF+Block-R):    {dedup_ratio_combined:.4f}x')
print(f'  Space saving:                {total_saved_combined/total_size*100:.2f}%')
print('=' * 60)

---
## 5. Summary & Comparison

In [ ]:
# Summary comparison table
summary_data = {
    'Method': [
        'No Dedup (decompressed)',
        'DupHunter GF-R (File-level)',
        'DupHunter GF+Block-R (File + 512KiB Block)',
    ],
    'Total Size (GiB)': [
        total_size / 1024/1024/1024,
        total_size / 1024/1024/1024,
        total_size / 1024/1024/1024,
    ],
    'Stored Size (GiB)': [
        total_size / 1024/1024/1024,
        unique_size / 1024/1024/1024,
        final_stored_size / 1024/1024/1024,
    ],
    'Saved (GiB)': [
        0,
        dedup_size / 1024/1024/1024,
        total_saved_combined / 1024/1024/1024,
    ],
    'Dedup Ratio': [
        1.0,
        dedup_ratio_gfr,
        dedup_ratio_combined,
    ],
    'Space Saving (%)': [
        0,
        dedup_size / total_size * 100,
        total_saved_combined / total_size * 100,
    ]
}

df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.round(4)
print(df_summary.to_string(index=False))

In [ ]:
# Visualization: Dedup Ratio Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

methods = ['No Dedup', 'DupHunter\nGF-R\n(File)', 'DupHunter\nGF+Block-R\n(File+512KiB)']
ratios = [1.0, dedup_ratio_gfr, dedup_ratio_combined]
savings = [0, dedup_size/total_size*100, total_saved_combined/total_size*100]

colors = ['#95a5a6', '#3498db', '#1a5276']

# Dedup ratio bar chart
bars1 = axes[0].bar(methods, ratios, color=colors, edgecolor='black', alpha=0.85)
axes[0].set_ylabel('Dedup Ratio (x)', fontsize=12)
axes[0].set_title('DupHunter: Dedup Ratio Comparison', fontsize=14)
axes[0].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars1, ratios):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{val:.2f}x', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Space saving bar chart
bars2 = axes[1].bar(methods, savings, color=colors, edgecolor='black', alpha=0.85)
axes[1].set_ylabel('Space Saving (%)', fontsize=12)
axes[1].set_title('DupHunter: Space Saving Comparison', fontsize=14)
for bar, val in zip(bars2, savings):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('./duphunter_dedup_comparison.png', dpi=150)
plt.show()

In [ ]:
# Stacked bar: breakdown of stored vs saved data
fig, ax = plt.subplots(figsize=(10, 6))

method_labels = ['No Dedup', 'DupHunter\nGF-R', 'DupHunter\nGF+Block-R']

stored_vals = [
    total_size/1024/1024/1024,
    unique_size/1024/1024/1024,
    final_stored_size/1024/1024/1024
]
saved_vals = [
    0,
    dedup_size/1024/1024/1024,
    total_saved_combined/1024/1024/1024
]

x = np.arange(len(method_labels))
width = 0.5

ax.bar(x, stored_vals, width, label='Stored Data', color='#3498db', edgecolor='black')
ax.bar(x, saved_vals, width, bottom=stored_vals, label='Saved (Deduplicated)',
       color='#e74c3c', alpha=0.6, edgecolor='black')

ax.set_ylabel('Size (GiB)', fontsize=12)
ax.set_title('DupHunter: Storage Breakdown', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(method_labels)
ax.legend()

for i, (s, sv) in enumerate(zip(stored_vals, saved_vals)):
    ax.text(i, s + sv + 0.1, f'{s:.2f} GiB\nstored', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('./duphunter_storage_breakdown.png', dpi=150)
plt.show()

---
## 6. Save All Results

In [ ]:
# Save all results for cross-notebook comparison
import json

results_dict = {
    'layer_directory': layer_directory,
    'extract_directory': extract_directory,
    'total_layers': len(all_layers),
    'total_extracted_files': len(df_files),
    'unique_files': len(unique_files),
    'total_size_bytes': int(total_size),
    'unique_size_bytes': int(unique_size),

    # GF-R (file-level)
    'gfr_dedup_ratio': float(dedup_ratio_gfr),
    'gfr_saved_bytes': int(dedup_size),
    'gfr_space_saving_pct': float(dedup_size / total_size * 100),

    # GF+Block-R (file + 512KiB block)
    'gf_block_dedup_ratio': float(dedup_ratio_combined),
    'gf_block_saved_bytes': int(total_saved_combined),
    'gf_block_stored_bytes': int(final_stored_size),
    'gf_block_space_saving_pct': float(total_saved_combined / total_size * 100),

    'block_size': BLOCK_SIZE,
    'total_blocks': len(df_blocks),
    'unique_blocks': int(df_blocks['hash'].nunique()),
}

with open('./duphunter_results.json', 'w') as f:
    json.dump(results_dict, f, indent=2)

df_summary.to_csv('./duphunter_summary.csv', index=False)

print('Results saved:')
print('  - duphunter_results.json')
print('  - duphunter_summary.csv')
print('  - duphunter_dedup_comparison.png')
print('  - duphunter_storage_breakdown.png')
print('  - duphunter_file_dedup_distribution.png')